In [ ]:
import math
import sys

sys.setrecursionlimit(10_000)

Transitions = dict[str, dict[str, float]]


# ─────────────────────────────────────────────
# 1. GREEDY SEARCH — ITERATIVE
# ─────────────────────────────────────────────
def greedy_iterative(
    transitions: Transitions,
    start: str,
    terminal: str = ".",
) -> list[str]:
    """
    Greedy decoder (iterative).

    At each step pick the single highest-probability next token.
    Tie-breaking: lexicographically smallest token among equals (deterministic).

    Time:  O(L · b)   L = sequence length, b = max branching factor
    Space: O(L)       only the output sequence is stored
    """
    sequence = [start]
    current = start

    while True:
        nexts = transitions.get(current, {})
        if not nexts:
            break  # dead end

        best_prob = max(nexts.values())
        # Among tokens with equal max probability, pick lex-smallest
        chosen = min(t for t, p in nexts.items() if p == best_prob)

        sequence.append(chosen)
        current = chosen
        if current == terminal:
            break

    return sequence


# ─────────────────────────────────────────────
# 2. GREEDY SEARCH — RECURSIVE
# ─────────────────────────────────────────────
def greedy_recursive(
    transitions: Transitions,
    start: str,
    terminal: str = ".",
) -> list[str]:
    """
    Greedy decoder (recursive).

    Identical semantics and tie-breaking to the iterative version.

    Time:  O(L · b)
    Space: O(L)  — call stack depth equals sequence length.
                   Raise sys.setrecursionlimit() for very long sequences.
    """
    def _step(current: str, acc: list[str]) -> list[str]:
        if current == terminal:
            return acc

        nexts = transitions.get(current, {})
        if not nexts:
            return acc  # dead end

        best_prob = max(nexts.values())
        chosen = min(t for t, p in nexts.items() if p == best_prob)

        return _step(chosen, acc + [chosen])

    return _step(start, [start])


# ─────────────────────────────────────────────
# 3. BEAM SEARCH
# ─────────────────────────────────────────────
def beam_search(
    transitions: Transitions,
    start: str,
    k: int = 3,
    terminal: str = ".",
) -> tuple[list[str], list[tuple[float, list[str]]]]:
    """
    Beam search decoder (BFS-style).

    Maintains k partial sequences ranked by cumulative log-probability.
    At each step, every active beam is expanded to all next tokens;
    the top-k candidates (by cumulative log-prob) survive.
    Beams that hit the terminal token or a dead end are moved to
    'completed' and no longer expanded.

    Tie-breaking: among equal cumulative log-probs, the sequence that
    is lexicographically smaller (compared element-by-element) wins.

    Args:
        transitions: token → {next_token: probability}
        start:       first token of every sequence
        k:           beam size
        terminal:    token that ends a sequence (default ".")

    Returns:
        best_seq   — highest log-prob completed sequence
        completed  — list of (log_prob, sequence), best-first

    Time:  O(L · k · b)   per full decode
    Space: O(k · L)        active beams; O(C · L) completed (C ≤ k·L worst case)
    """
    # Each beam: (neg_log_prob, sequence)
    # We minimise neg_log_prob, which is equivalent to maximising log_prob.
    beams: list[tuple[float, list[str]]] = [(0.0, [start])]
    completed: list[tuple[float, list[str]]] = []

    while beams:
        candidates: list[tuple[float, list[str]]] = []

        for neg_lp, seq in beams:
            current = seq[-1]

            if current == terminal:
                completed.append((neg_lp, seq))
                continue

            nexts = transitions.get(current, {})
            if not nexts:
                completed.append((neg_lp, seq))  # dead end
                continue

            for token, prob in nexts.items():
                safe_prob = max(prob, 1e-12)      # guard against log(0)
                new_neg_lp = neg_lp - math.log(safe_prob)
                candidates.append((new_neg_lp, seq + [token]))

        if not candidates:
            break

        # Keep top-k; ties broken by lex order of the sequence
        candidates.sort(key=lambda x: (x[0], x[1]))
        beams = candidates[:k]

        # Drain newly-completed beams
        # active = []
        # for neg_lp, seq in beams:
        #     if seq[-1] == terminal or seq[-1] not in transitions or not transitions[seq[-1]]:
        #         completed.append((neg_lp, seq))
        #     else:
        #         active.append((neg_lp, seq))
        # beams = active

    # Sort completed: best (lowest neg_log_prob) first; lex tie-break
    completed.sort(key=lambda x: (x[0], x[1]))

    best_seq = completed[0][1] if completed else [start]
    # Convert stored neg_log_prob back to log_prob for the caller
    completed_out = [(-neg_lp, seq) for neg_lp, seq in completed]

    return best_seq, completed_out


In [2]:
if __name__ == "__main__":
    transitions: Transitions = {
        "I":      {"love": 0.7, "hate": 0.2, "see": 0.1},
        "love":   {"cats": 0.6, "dogs": 0.4},
        "hate":   {"snakes": 0.9, "bugs": 0.1},
        "see":    {"you": 1.0},
        "cats":   {".": 1.0},
        "dogs":   {".": 1.0},
        "snakes": {".": 1.0},
        "bugs":   {".": 1.0},
        "you":    {".": 1.0},
    }

    gi = greedy_iterative(transitions, "I")
    gr = greedy_recursive(transitions, "I")
    best, all_completed = beam_search(transitions, "I", k=3)

    print("Greedy (iterative):", " → ".join(gi))
    print("Greedy (recursive):", " → ".join(gr))
    print("Beam best:         ", " → ".join(best))
    print("All completed beams:")
    for lp, seq in all_completed:
        print(f"  log-p={lp:.3f}  {' → '.join(seq)}")

Greedy (iterative): I → love → cats → .
Greedy (recursive): I → love → cats → .
Beam best:          I → love → cats → .
All completed beams:
  log-p=-0.868  I → love → cats → .
  log-p=-1.273  I → love → dogs → .
  log-p=-1.715  I → hate → snakes → .


In [ ]:
transitions: Transitions = {
    "I":      {"love": 0.7, "hate": 0.2, "see": 0.1},
    "love":   {"cats": 0.6, "dogs": 0.4},
    "hate":   {"snakes": 0.9, "bugs": 0.1},
    "see":    {"you": 1.0},
    "cats":   {".": 1.0},
    "dogs":   {".": 1.0},
    "snakes": {".": 1.0},
    "bugs":   {".": 1.0},
    "you":    {".": 1.0},
}


    def _step(current: str, acc: list[str]) -> list[str]:
        if current == terminal:
            return acc

        nexts = transitions.get(current, {})
        if not nexts:
            return acc  # dead end

        best_prob = max(nexts.values())
        chosen = min(t for t, p in nexts.items() if p == best_prob)

        return _step(chosen, acc + [chosen])

    return _step(start, [start])

